In [16]:
pip -q install langchain langchain-core langchain-groq langgraph PyPDF2 Pillow

In [17]:
import os
from getpass import getpass
from google.colab import userdata

# Get Groq API key from user
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

print("✓ Groq API key configured!")
print(f"✓ Using model: mixtral-8x7b-32768")

✓ Groq API key configured!
✓ Using model: mixtral-8x7b-32768


In [18]:
import json
from typing import Optional, TypedDict, List, Dict, Any
from datetime import datetime
import re
from pathlib import Path

# LangChain imports
from langchain_core.messages import BaseMessage
from pydantic import BaseModel, Field

# LangGraph imports
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Groq LLM
from langchain_groq import ChatGroq

# PDF processing
import PyPDF2
from PIL import Image

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


In [19]:
# Data Models
import sqlite3
import hashlib
import uuid

class AuditLog(BaseModel):
    """Audit trail entry"""
    audit_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    timestamp: str = Field(default_factory=lambda: datetime.now().isoformat())
    document_id: str = Field(..., description="Document ID")
    operation_type: str = Field(..., description="Type of operation (LOAD, CLASSIFY, EXTRACT, VALIDATE, ROUTE, STORE, ARCHIVE, REVIEW)")
    operation_status: str = Field(..., description="SUCCESS, FAILURE, WARNING, REVIEW_REQUIRED")
    explanation: str = Field(..., description="Detailed explanation of operation")
    input_data: Dict[str, Any] = Field(default_factory=dict, description="Input to operation")
    output_data: Dict[str, Any] = Field(default_factory=dict, description="Output from operation")
    confidence_score: float = Field(default=0.0)
    data_hash: str = Field(..., description="SHA256 hash for data integrity")
    user_id: str = Field(default="SYSTEM")
    compliance_notes: str = Field(default="")


class CeaseDesistExtraction(BaseModel):
    """Extracted data from Cease & Desist documents"""
    sender: str = Field(..., description="Entity sending cease & desist")
    recipient: str = Field(..., description="Entity receiving cease & desist")
    violation_type: str = Field(..., description="Type of violation (Copyright, Patent, Trademark, Trade Secret, etc.)")
    details: str = Field(..., description="Details of alleged violation")
    cease_by_date: str = Field(..., description="Date by which activity must cease")
    consequences: str = Field(..., description="Consequences of non-compliance")
    key_clauses: List[str] = Field(default_factory=list, description="Important clauses")
    signatures: List[str] = Field(default_factory=list, description="Signature locations")


class DocumentMetadata(BaseModel):
    """Metadata about processed documents"""
    document_id: str = Field(..., description="Unique document identifier")
    document_type: str = Field(..., description="Type of document (LOA, Notice, Business, Cease)")
    classification: str = Field(..., description="Classification (Cease, Uncertain, Irrelevant)")
    processing_date: str = Field(..., description="Date document was processed")
    confidence_score: float = Field(..., description="Extraction confidence (0-100)")
    file_path: Optional[str] = Field(None, description="Original file path")
    page_count: Optional[int] = Field(None, description="Number of pages")
    received_date: Optional[str] = Field(None, description="Date document was received")


class LOAExtraction(BaseModel):
    """Extracted data from Letter of Authorization"""
    authorizing_party: str = Field(..., description="Entity granting authorization")
    authorized_party: str = Field(..., description="Entity receiving authorization")
    authorization_scope: str = Field(..., description="What is authorized")
    effective_date: str = Field(..., description="When authorization starts")
    expiration_date: Optional[str] = Field(None, description="When authorization ends")
    signatures: List[str] = Field(default_factory=list, description="Signature locations")
    key_clauses: List[str] = Field(default_factory=list, description="Important clauses")


class NoticeExtraction(BaseModel):
    """Extracted data from Notice documents"""
    notice_type: str = Field(..., description="Type of notice")
    recipient: str = Field(..., description="Who is receiving the notice")
    sender: Optional[str] = Field(None, description="Who sent the notice")
    subject: str = Field(..., description="Notice subject")
    issue_date: str = Field(..., description="Date notice was issued")
    action_required: str = Field(..., description="What action is required")
    important_dates: Dict[str, str] = Field(default_factory=dict, description="Key dates")


class BusinessDocExtraction(BaseModel):
    """Extracted data from Business Documents"""
    doc_type: str = Field(..., description="Type of business document")
    parties_involved: List[str] = Field(..., description="Parties in document")
    key_terms: Dict[str, str] = Field(default_factory=dict, description="Important terms")
    amounts: List[float] = Field(default_factory=list, description="Financial amounts")
    dates: Dict[str, str] = Field(default_factory=dict, description="Important dates")
    key_clauses: List[str] = Field(default_factory=list, description="Important clauses")


class ValidationResult(BaseModel):
    """Result of data validation"""
    is_valid: bool = Field(..., description="Whether data passed validation")
    confidence_score: float = Field(..., description="Confidence in extraction (0-100)")
    issues_found: List[str] = Field(default_factory=list, description="Validation issues")
    requires_review: bool = Field(..., description="Whether human review is needed")
    recommendations: List[str] = Field(default_factory=list, description="Improvement suggestions")


class DocumentProcessingResult(BaseModel):
    """Complete result of document processing"""
    metadata: DocumentMetadata
    document_type: str
    classification: str
    extracted_data: Dict[str, Any]
    validation: ValidationResult
    raw_text: str
    processing_log: List[str]
    audit_trail: List[AuditLog] = Field(default_factory=list)
    routing_decision: str = Field(default="PENDING")


class ProcessingState(TypedDict):
    """State for LangGraph workflow"""
    document_id: str
    file_path: str
    document_text: str
    document_type: str
    document_classification: str  # Cease, Uncertain, Irrelevant
    document_type_confidence: float
    extracted_data: Dict[str, Any]
    validation_result: Optional[ValidationResult]
    confidence_score: float
    processing_log: List[str]
    audit_trail: List[AuditLog]
    requires_human_review: bool
    human_review_result: Optional[Dict[str, Any]]
    final_result: Optional[DocumentProcessingResult]
    messages: List[BaseMessage]
    is_approved: bool
    received_date: str

print("✓ Data models defined successfully!")

✓ Data models defined successfully!


In [20]:
class AuditTrailManager:
    """Manages comprehensive audit trails"""
    def __init__(self):
        self.audit_logs: List[AuditLog] = []

    def create_audit_entry(self, doc_id: str, operation_type: str, status: str,
                          explanation: str, input_data: Dict = None, output_data: Dict = None,
                          confidence: float = 0.0, compliance_notes: str = "") -> AuditLog:
        """Create and log audit entry"""
        # Handle None inputs properly for dictionary unpacking
        input_dict = input_data if input_data is not None else {}
        output_dict = output_data if output_data is not None else {}
        data_to_hash = json.dumps({**input_dict, **output_dict})
        data_hash = hashlib.sha256(data_to_hash.encode()).hexdigest()

        audit_log = AuditLog(
            document_id=doc_id,
            operation_type=operation_type,
            operation_status=status,
            explanation=explanation,
            input_data=input_dict,
            output_data=output_dict,
            confidence_score=confidence,
            data_hash=data_hash,
            compliance_notes=compliance_notes
        )

        self.audit_logs.append(audit_log)
        return audit_log

    def get_audit_trail(self, doc_id: str) -> List[AuditLog]:
        """Get all audit entries for a document"""
        return [log for log in self.audit_logs if log.document_id == doc_id]

    def print_audit_trail(self, doc_id: str):
        """Print formatted audit trail"""
        entries = self.get_audit_trail(doc_id)
        print(f"\n{'='*70}")
        print(f"AUDIT TRAIL FOR: {doc_id}")
        print(f"{'='*70}")
        for entry in entries:
            print(f"\n[{entry.timestamp}] {entry.operation_type}")
            print(f"  Status: {entry.operation_status}")
            print(f"  Explanation: {entry.explanation}")
            print(f"  Confidence: {entry.confidence_score:.1f}%")
            print(f"  Data Hash: {entry.data_hash[:16]}...")

In [21]:
# ========================================================================
# HELPER EXTRACTION FUNCTIONS (Additional)
# ========================================================================

def _extract_party(text: str, pattern_prefix: str) -> str:
    """Extract party information from text"""
    # Look for capitalized names or organization patterns
    patterns = [
        rf'{pattern_prefix}\s+([A-Z][A-Za-z\s&,\.]+?)(?:[,\.]|\s(?:on|the|by|and))',
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    # Fallback: extract first capitalized phrase
    match = re.search(r'[A-Z][A-Za-z\s]+(?:Inc|LLC|Corp|Ltd|Co|Company|Corporation)?', text)
    if match:
        return match.group(0).strip()

    return "Unknown"


def _extract_date(text: str, pattern_prefix: str) -> str:
    """Extract date from text"""
    date_patterns = [
        r'\b(\d{4}[-/]\d{2}[-/]\d{2})\b',  # YYYY-MM-DD
        r'\b(\d{1,2}[-/]\d{1,2}[-/]\d{4})\b',  # MM-DD-YYYY
        r'\b((?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4})\b',
        r'\b(\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{4})\b',
    ]

    # Look for dates near the pattern
    for date_pattern in date_patterns:
        match = re.search(date_pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    return "Date not specified"


def _extract_subject(text: str) -> str:
    """Extract subject line or main topic"""
    # Look for "Subject:", "Re:", "RE:" patterns
    match = re.search(r'(?:Subject|Re|RE|Regarding)[:\s]+([^\n]{20,100})', text, re.IGNORECASE)
    if match:
        return match.group(1).strip()

    # Fallback: first substantial line
    lines = text.split('\n')
    for line in lines:
        if 20 < len(line) < 200:
            return line.strip()

    return "No subject found"


def _extract_clauses(text: str) -> list:
    """Extract important clauses/sections"""
    clauses = []

    # Look for numbered sections, bullet points, or clause headers
    clause_patterns = [
        r'(?:Section|Clause|Article)\s+(\d+)[:\s]+([^\n]{20,100})',
        r'^[-•*]\s+([^\n]{20,100})',
        r'(?:Clause|Section|Article)[:\s]+([^\n]{20,100})',
    ]

    for pattern in clause_patterns:
        matches = re.finditer(pattern, text, re.IGNORECASE | re.MULTILINE)
        for match in matches:
            clause_text = match.group(1) if match.lastindex >= 1 else match.group(0)
            if clause_text and len(clause_text) > 10:
                clauses.append(clause_text.strip()[:100])

    return clauses[:5]  # Return top 5 clauses


def _extract_signatures(text: str) -> list:
    """Extract signature locations/blocks"""
    signatures = []

    # Look for signature lines
    sig_patterns = [
        r'(?:Signed|Signature)[:\s]+(.+)',
        r'(?:By|Sign here)[:\s]+(.+)',
        r'_+\s+',  # Lines with underscores (signature blanks)
    ]

    for pattern in sig_patterns:
        matches = re.finditer(pattern, text, re.IGNORECASE)
        for match in matches:
            if match.lastindex:
                signatures.append(f"Signature: {match.group(1).strip()[:50]}")
            else:
                signatures.append("Signature block found")

    return signatures[:3]  # Return top 3 signatures


def _generate_recommendations(confidence: float, issues: list) -> list:
    """Generate recommendations based on validation results"""
    recommendations = []

    if confidence < 60:
        recommendations.append("Recommend manual review due to low confidence score")

    if len(issues) > 0:
        recommendations.append(f"Address {len(issues)} validation issues identified")

    if confidence >= 80:
        recommendations.append("Data quality is good - proceed with processing")

    if "Missing field" in str(issues):
        recommendations.append("Request missing information from document sender")

    return recommendations


# ========================================================================
# AGENT CLASSES FOR ROUTING
# ========================================================================

class DatabaseAgent:
    """Handles storage of Cease & Desist documents"""

    def __init__(self, db_path: str = "./cease_desist_documents.db"):
        self.db_path = db_path
        self._initialize_database()

    def _initialize_database(self):
        """Initialize SQLite database"""
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()

            cursor.execute('''
                CREATE TABLE IF NOT EXISTS cease_documents (
                    doc_id TEXT PRIMARY KEY,
                    received_date TEXT,
                    doc_name TEXT,
                    sender TEXT,
                    recipient TEXT,
                    violation_type TEXT,
                    details TEXT,
                    cease_by_date TEXT,
                    consequences TEXT,
                    confidence REAL,
                    audit_id TEXT,
                    stored_date TEXT,
                    raw_data TEXT
                )
            ''')
            conn.commit()
            conn.close()
        except Exception as e:
            print(f"⚠ Database initialization warning: {str(e)}")

    def store_cease_document(self, doc_id: str, received_date: str, doc_name: str,
                            extracted: dict, confidence: float, audit_id: str) -> dict:
        """Store a cease document in database"""
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()

            cursor.execute('''
                INSERT OR REPLACE INTO cease_documents
                (doc_id, received_date, doc_name, sender, recipient, violation_type,
                 details, cease_by_date, consequences, confidence, audit_id, stored_date, raw_data)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                doc_id,
                received_date,
                doc_name,
                extracted.get("sender", "Unknown"),
                extracted.get("recipient", "Unknown"),
                extracted.get("violation_type", "Unknown"),
                extracted.get("details", "")[:500],
                extracted.get("cease_by_date", ""),
                extracted.get("consequences", "")[:500],
                confidence,
                audit_id,
                datetime.now().isoformat(),
                json.dumps(extracted)
            ))

            conn.commit()
            conn.close()

            return {
                "status": "STORED",
                "message": f"✓ Cease document stored in database",
                "database": self.db_path,
                "record_count": 1
            }
        except Exception as e:
            return {
                "status": "FAILED",
                "message": f"✗ Failed to store document: {str(e)}",
                "error": str(e)
            }


class ArchivingAgent:
    """Handles archiving of irrelevant documents"""

    def __init__(self, archive_path: str = "./irrelevant_archive"):
        self.archive_path = Path(archive_path)
        self.archive_path.mkdir(exist_ok=True)

    def archive_document(self, doc_id: str, received_date: str, doc_name: str,
                        doc_text: str, audit_id: str) -> dict:
        """Archive an irrelevant document"""
        try:
            # Create archive entry
            archive_file = self.archive_path / f"{doc_id}_{doc_name}.txt"

            archive_content = f"""
ARCHIVED DOCUMENT - IRRELEVANT
==============================
Document ID: {doc_id}
Received Date: {received_date}
Original Name: {doc_name}
Audit ID: {audit_id}
Archived Date: {datetime.now().isoformat()}

CONTENT:
--------
{doc_text[:1000]}
"""

            with open(archive_file, 'w', encoding='utf-8') as f:
                f.write(archive_content)

            return {
                "status": "ARCHIVED",
                "message": f"✓ Document archived to {archive_file.name}",
                "archive_location": str(archive_file),
                "archive_path": str(self.archive_path)
            }
        except Exception as e:
            return {
                "status": "FAILED",
                "message": f"✗ Failed to archive document: {str(e)}",
                "error": str(e)
            }


class HITLReviewAgent:
    """Handles human-in-the-loop review of uncertain documents"""

    def __init__(self, review_path: str = "./human_review_queue"):
        self.review_path = Path(review_path)
        self.review_path.mkdir(exist_ok=True)

    def queue_for_review(self, doc_id: str, received_date: str, doc_name: str,
                        extracted: dict, confidence: float, audit_id: str) -> dict:
        """Queue document for human review"""
        try:
            # Create review entry
            review_file = self.review_path / f"REVIEW_{doc_id}_{doc_name}.json"

            review_data = {
                "doc_id": doc_id,
                "received_date": received_date,
                "original_name": doc_name,
                "audit_id": audit_id,
                "queued_date": datetime.now().isoformat(),
                "confidence": confidence,
                "extracted_data": extracted,
                "review_status": "PENDING",
                "instructions": "Please review this document and determine if it is a valid Cease & Desist"
            }

            with open(review_file, 'w', encoding='utf-8') as f:
                json.dump(review_data, f, indent=2, default=str)

            return {
                "status": "QUEUED",
                "message": f"✓ Document queued for human review",
                "review_location": str(review_file),
                "review_path": str(self.review_path)
            }
        except Exception as e:
            return {
                "status": "FAILED",
                "message": f"✗ Failed to queue document for review: {str(e)}",
                "error": str(e)
            }


# ========================================================================
# INSTANTIATE AUDIT MANAGER AND AGENTS
# ========================================================================

# Create global instances of audit manager and agents
audit_manager = AuditTrailManager()
database_agent = DatabaseAgent()
archiving_agent = ArchivingAgent()
hitl_agent = HITLReviewAgent()

print("✓ Audit manager initialized")
print("✓ Database agent initialized")
print("✓ Archiving agent initialized")
print("✓ Human-in-the-loop review agent initialized")
print("\n✅ All helper functions and agents ready for document processing!")


✓ Audit manager initialized
✓ Database agent initialized
✓ Archiving agent initialized
✓ Human-in-the-loop review agent initialized

✅ All helper functions and agents ready for document processing!


In [22]:
def load_document(file_path: str) -> str:
    """
    Load and preprocess documents (PDF or image).

    Args:
        file_path: Path to PDF or image file

    Returns:
        Extracted text from document
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    text = ""

    # Handle PDF files
    if file_path.suffix.lower() == '.pdf':
        try:
            with open(file_path, 'rb') as pdf_file:
                pdf_reader = PyPDF2.PdfReader(pdf_file)
                for page_num, page in enumerate(pdf_reader.pages):
                    text += f"\n--- Page {page_num + 1} ---\n"
                    text += page.extract_text()
        except Exception as e:
            raise ValueError(f"Error loading PDF: {str(e)}")

    # Handle image files
    elif file_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.gif']:
        try:
            image = Image.open(file_path)
            text = f"Image file: {file_path.name} (OCR not available in demo)"
        except Exception as e:
            raise ValueError(f"Error loading image: {str(e)}")

    # Handle text files
    elif file_path.suffix.lower() in ['.txt', '.md']:
        with open(file_path, 'r', encoding='utf-8') as text_file:
            text = text_file.read()
    else:
        raise ValueError(f"Unsupported file format: {file_path.suffix}")

    return text if text.strip() else "No text could be extracted from the document."


def classify_document(text: str) -> Dict[str, Any]:
    """
    Classify document into: Cease, Uncertain, or Irrelevant

    Args:
        text: Document text to classify

    Returns:
        Classification result with category and confidence
    """
    text_lower = text.lower()

    # CEASE & DESIST indicators (high confidence)
    cease_keywords = [
        'cease and desist', 'cease & desist', 'cease or desist',
        'immediately cease', 'discontinue', 'stop immediately',
        'infringement', 'violation', 'unauthorized use',
        'intellectual property', 'copyright', 'trademark', 'patent',
        'trade secret', 'confidential information',
        'demand for', 'cease', 'infringer'
    ]

    # UNCERTAIN indicators (medium confidence)
    uncertain_keywords = [
        'notice', 'legal notice', 'formal notice', 'warning',
        'letter of concern', 'objection', 'complaint',
        'unauthorized', 'dispute', 'claim'
    ]

    # Calculate keyword scores
    cease_score = sum(1 for keyword in cease_keywords if keyword in text_lower)
    uncertain_score = sum(1 for keyword in uncertain_keywords if keyword in text_lower)

    # Additional cease markers
    cease_markers = [
        bool(re.search(r'\bstop\b.*\bimmediately', text_lower)),
        bool(re.search(r'cease.*all.*activities', text_lower)),
        bool(re.search(r'(?:copyright|patent|trademark|trade secret).*(?:infringement|violation)', text_lower)),
        bool(re.search(r'demand.*(?:cease|stop|discontinue)', text_lower)),
    ]
    cease_score += sum(cease_markers)

    # Determine classification
    if cease_score >= 3:
        classification = "Cease"
        confidence = min(100, (cease_score / 6) * 100)
        status_explanation = "Document identified as valid Cease & Desist request"
    elif cease_score >= 2 or (cease_score >= 1 and uncertain_score >= 2):
        classification = "Uncertain"
        confidence = min(100, (cease_score + uncertain_score) / 8 * 100)
        status_explanation = "Document may be a Cease & Desist but confidence is moderate - requires manual review"
    else:
        classification = "Irrelevant"
        confidence = min(100, max(100 - (cease_score + uncertain_score) * 10, 50))
        status_explanation = "Document does not appear to be a Cease & Desist request"

    return {
        'classification': classification,
        'confidence': confidence,
        'cease_score': cease_score,
        'uncertain_score': uncertain_score,
        'status_explanation': status_explanation,
        'reasoning': f"Classification: {classification} | Score: {cease_score} cease indicators | Confidence: {confidence:.1f}%"
    }


def extract_cease_document(text: str) -> Dict[str, Any]:
    """
    Extract structured information from Cease & Desist document

    Args:
        text: Document text

    Returns:
        Structured extracted information
    """
    extracted = {
        'sender': _extract_party(text, '(?:from|by|sent by|issued by)'),
        'recipient': _extract_party(text, '(?:to|against|regarding|concerning)'),
        'violation_type': _extract_violation_type(text),
        'details': _extract_violation_details(text),
        'cease_by_date': _extract_date(text, '(?:cease by|stop by|by|by which|no later than)'),
        'consequences': _extract_consequences(text),
        'key_clauses': _extract_clauses(text),
        'signatures': _extract_signatures(text)
    }
    return extracted


def _extract_violation_type(text: str) -> str:
    """Extract type of violation"""
    violation_types = {
        'Copyright': r'copyright',
        'Patent': r'patent',
        'Trademark': r'trademark',
        'Trade Secret': r'trade secret',
        'Intellectual Property': r'intellectual property',
        'Defamation': r'defam|slander|libel',
        'Breach of Contract': r'breach.*contract',
    }
    for vtype, pattern in violation_types.items():
        if re.search(pattern, text, re.IGNORECASE):
            return vtype
    return "Unspecified Violation"


def _extract_violation_details(text: str) -> str:
    """Extract details of alleged violation"""
    # Look for detailed description after common markers
    markers = ['violation', 'infringement', 'unauthorized use', 'details', 'regarding']
    for marker in markers:
        match = re.search(f'{marker}[:.\\s]*([^.!?]{50,300})', text, re.IGNORECASE)
        if match:
            return match.group(1).strip()
    # Fall back to extracting substantial paragraph
    paragraphs = text.split('\n\n')
    for para in paragraphs:
        if len(para) > 100:
            return para[:200].strip()
    return "No specific details found"


def _extract_consequences(text: str) -> str:
    """Extract consequences of non-compliance"""
    consequence_patterns = [
        r'(?:failure|nonc|failure to comply|if you do not).*?(?:will result in|consequences|legal action|lawsuit)[:.\\s]*([^.!?]{30,200})',
        r'(?:consequences|penalties|damages)[:.\\s]*([^.!?]{30,200})',
        r'(?:we will|will be forced to|we intend to).*?(?:legal|court|lawsuit|damages)[:.\\s]*([^.!?]{40,200})',
    ]

    for pattern in consequence_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return "Consequences not explicitly stated"

print("✓ Classification and extraction functions ready!")

✓ Classification and extraction functions ready!


In [23]:

# ========================================================================
# PDF LOADER TOOL
# ========================================================================
class PDFLoaderTool:
    """Tool for loading and extracting text from PDF documents"""

    def __init__(self):
        self.supported_formats = ['.pdf']

    def load_pdf(self, file_path: str) -> Dict[str, Any]:
        """
        Load and extract text from PDF file

        Args:
            file_path: Path to PDF file

        Returns:
            Dictionary with extracted text and metadata
        """
        try:
            file_path = Path(file_path)

            if not file_path.exists():
                return {
                    "status": "FAILED",
                    "error": f"File not found: {file_path}",
                    "text": "",
                    "page_count": 0
                }

            if file_path.suffix.lower() != '.pdf':
                return {
                    "status": "FAILED",
                    "error": f"Not a PDF file: {file_path.suffix}",
                    "text": "",
                    "page_count": 0
                }

            text = ""
            page_count = 0

            with open(file_path, 'rb') as pdf_file:
                pdf_reader = PyPDF2.PdfReader(pdf_file)
                page_count = len(pdf_reader.pages)

                for page_num, page in enumerate(pdf_reader.pages):
                    text += f"\n--- Page {page_num + 1} ---\n"
                    text += page.extract_text()

            return {
                "status": "SUCCESS",
                "text": text if text.strip() else "No text extracted",
                "page_count": page_count,
                "file_path": str(file_path),
                "file_size": file_path.stat().st_size,
                "text_length": len(text)
            }

        except Exception as e:
            return {
                "status": "FAILED",
                "error": f"PDF loading error: {str(e)}",
                "text": "",
                "page_count": 0
            }


# ========================================================================
# OCR TOOL
# ========================================================================
class OCRTool:
    """Tool for Optical Character Recognition on image documents"""

    def __init__(self):
        self.supported_formats = ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff']
        self.use_fallback = True  # Use fallback when pytesseract not available

    def extract_text_from_image(self, file_path: str) -> Dict[str, Any]:
        """
        Extract text from image file using OCR

        Args:
            file_path: Path to image file

        Returns:
            Dictionary with extracted text and metadata
        """
        try:
            file_path = Path(file_path)

            if not file_path.exists():
                return {
                    "status": "FAILED",
                    "error": f"File not found: {file_path}",
                    "text": "",
                    "confidence": 0.0
                }

            if file_path.suffix.lower() not in self.supported_formats:
                return {
                    "status": "FAILED",
                    "error": f"Unsupported image format: {file_path.suffix}",
                    "text": "",
                    "confidence": 0.0
                }

            image = Image.open(file_path)
            image_size = image.size

            # Try to use pytesseract if available, otherwise use fallback
            try:
                import pytesseract
                text = pytesseract.image_to_string(image)
                confidence = 0.9 if text.strip() else 0.1
                method = "pytesseract"
            except ImportError:
                if self.use_fallback:
                    text = f"[OCR Fallback] Image file: {file_path.name}, Size: {image_size}, Format: {image.format}"
                    confidence = 0.3
                    method = "fallback"
                else:
                    return {
                        "status": "FAILED",
                        "error": "pytesseract not installed and fallback disabled",
                        "text": "",
                        "confidence": 0.0
                    }

            return {
                "status": "SUCCESS",
                "text": text if text.strip() else "[No text detected in image]",
                "confidence": confidence,
                "file_path": str(file_path),
                "image_size": image_size,
                "image_format": image.format,
                "ocr_method": method,
                "text_length": len(text)
            }

        except Exception as e:
            return {
                "status": "FAILED",
                "error": f"OCR error: {str(e)}",
                "text": "",
                "confidence": 0.0
            }


# ========================================================================
# CLASSIFICATION TOOL
# ========================================================================
class ClassificationTool:
    """Tool for classifying documents into categories"""

    def __init__(self):
        self.categories = ["Cease", "Uncertain", "Irrelevant"]

    def classify(self, text: str) -> Dict[str, Any]:
        """
        Classify document text

        Args:
            text: Document text to classify

        Returns:
            Dictionary with classification and confidence
        """
        try:
            text_lower = text.lower()

            # Cease & Desist keywords
            cease_keywords = [
                'cease and desist', 'cease & desist', 'cease or desist',
                'immediately cease', 'discontinue', 'stop immediately',
                'infringement', 'violation', 'unauthorized use',
                'intellectual property', 'copyright', 'trademark', 'patent',
                'trade secret', 'confidential information',
                'demand for', 'cease', 'infringer'
            ]

            # Uncertain keywords
            uncertain_keywords = [
                'notice', 'legal notice', 'formal notice', 'warning',
                'letter of concern', 'objection', 'complaint',
                'unauthorized', 'dispute', 'claim'
            ]

            cease_score = sum(1 for keyword in cease_keywords if keyword in text_lower)
            uncertain_score = sum(1 for keyword in uncertain_keywords if keyword in text_lower)

            # Additional markers
            cease_markers = [
                bool(re.search(r'\bstop\b.*\bimmediately', text_lower)),
                bool(re.search(r'cease.*all.*activities', text_lower)),
                bool(re.search(r'(?:copyright|patent|trademark|trade secret).*(?:infringement|violation)', text_lower)),
                bool(re.search(r'demand.*(?:cease|stop|discontinue)', text_lower)),
            ]
            cease_score += sum(cease_markers)

            # Determine classification
            if cease_score >= 3:
                classification = "Cease"
                confidence = min(100, (cease_score / 6) * 100)
            elif cease_score >= 2 or (cease_score >= 1 and uncertain_score >= 2):
                classification = "Uncertain"
                confidence = min(100, (cease_score + uncertain_score) / 8 * 100)
            else:
                classification = "Irrelevant"
                confidence = min(100, max(100 - (cease_score + uncertain_score) * 10, 50))

            return {
                "status": "SUCCESS",
                "classification": classification,
                "confidence": confidence,
                "cease_score": cease_score,
                "uncertain_score": uncertain_score,
                "reasoning": f"Classification: {classification} | Cease Score: {cease_score} | Confidence: {confidence:.1f}%"
            }

        except Exception as e:
            return {
                "status": "FAILED",
                "error": f"Classification error: {str(e)}",
                "classification": "Uncertain",
                "confidence": 0.0
            }


# ========================================================================
# EXTRACTION TOOL
# ========================================================================
class ExtractionTool:
    """Tool for extracting structured information from documents"""

    def __init__(self):
        self.extraction_methods = ["regex", "pattern", "keyword"]

    def extract_cease_document(self, text: str) -> Dict[str, Any]:
        """
        Extract structured information from Cease & Desist document

        Args:
            text: Document text

        Returns:
            Dictionary with extracted fields
        """
        try:
            extracted = {
                'sender': self._extract_party(text, '(?:from|by|sent by|issued by)'),
                'recipient': self._extract_party(text, '(?:to|against|regarding|concerning)'),
                'violation_type': self._extract_violation_type(text),
                'details': self._extract_violation_details(text),
                'cease_by_date': self._extract_date(text, '(?:cease by|stop by|by|by which|no later than)'),
                'consequences': self._extract_consequences(text),
                'key_clauses': self._extract_clauses(text),
                'signatures': self._extract_signatures(text)
            }

            return {
                "status": "SUCCESS",
                "extracted_data": extracted,
                "fields_extracted": len([v for v in extracted.values() if v]),
                "extraction_method": "regex"
            }

        except Exception as e:
            return {
                "status": "FAILED",
                "error": f"Extraction error: {str(e)}",
                "extracted_data": {},
                "fields_extracted": 0
            }

    def extract_generic_document(self, text: str, doc_classification: str = None) -> Dict[str, Any]:
        """
        Extract information from any document

        Args:
            text: Document text
            doc_classification: Optional document classification

        Returns:
            Dictionary with extracted fields
        """
        try:
            if doc_classification == "Cease":
                return self.extract_cease_document(text)

            # Generic extraction for all document types
            extracted = {
                'sender': self._extract_party(text, 'from|by'),
                'recipient': self._extract_party(text, 'to'),
                'subject': self._extract_subject(text),
                'key_clauses': self._extract_clauses(text),
                'dates': self._extract_all_dates(text)
            }

            return {
                "status": "SUCCESS",
                "extracted_data": extracted,
                "fields_extracted": len([v for v in extracted.values() if v]),
                "extraction_method": "generic"
            }

        except Exception as e:
            return {
                "status": "FAILED",
                "error": f"Generic extraction error: {str(e)}",
                "extracted_data": {},
                "fields_extracted": 0
            }

    def _extract_party(self, text: str, pattern_prefix: str) -> str:
        """Extract party information"""
        patterns = [
            rf'{pattern_prefix}\s+([A-Z][A-Za-z\s&,\.]+?)(?:[,\.]|\s(?:on|the|by|and))',
        ]
        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                return match.group(1).strip()
        match = re.search(r'[A-Z][A-Za-z\s]+(?:Inc|LLC|Corp|Ltd|Co|Company|Corporation)?', text)
        return match.group(0).strip() if match else "Unknown"

    def _extract_date(self, text: str, pattern_prefix: str) -> str:
        """Extract date from text"""
        date_patterns = [
            r'\b(\d{4}[-/]\d{2}[-/]\d{2})\b',
            r'\b(\d{1,2}[-/]\d{1,2}[-/]\d{4})\b',
            r'\b((?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4})\b',
            r'\b(\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{4})\b',
        ]
        for date_pattern in date_patterns:
            match = re.search(date_pattern, text, re.IGNORECASE)
            if match:
                return match.group(1).strip()
        return "Date not specified"

    def _extract_all_dates(self, text: str) -> Dict[str, str]:
        """Extract all dates found in document"""
        dates = {}
        date_pattern = r'\b(\d{4}[-/]\d{2}[-/]\d{2})\b'
        matches = re.finditer(date_pattern, text)
        for i, match in enumerate(matches, 1):
            dates[f"date_{i}"] = match.group(1)
        return dates

    def _extract_subject(self, text: str) -> str:
        """Extract subject line"""
        match = re.search(r'(?:Subject|Re|RE|Regarding)[:\s]+([^\n]{20,100})', text, re.IGNORECASE)
        if match:
            return match.group(1).strip()
        lines = text.split('\n')
        for line in lines:
            if 20 < len(line) < 200:
                return line.strip()
        return "No subject found"

    def _extract_violation_type(self, text: str) -> str:
        """Extract type of violation"""
        violation_types = {
            'Copyright': r'copyright',
            'Patent': r'patent',
            'Trademark': r'trademark',
            'Trade Secret': r'trade secret',
            'Intellectual Property': r'intellectual property',
            'Defamation': r'defam|slander|libel',
            'Breach of Contract': r'breach.*contract',
        }
        for vtype, pattern in violation_types.items():
            if re.search(pattern, text, re.IGNORECASE):
                return vtype
        return "Unspecified Violation"

    def _extract_violation_details(self, text: str) -> str:
        """Extract details of violation"""
        markers = ['violation', 'infringement', 'unauthorized use', 'details', 'regarding']
        for marker in markers:
            match = re.search(f'{marker}[:.\\s]*([^.!?]{50,300})', text, re.IGNORECASE)
            if match:
                return match.group(1).strip()
        paragraphs = text.split('\n\n')
        for para in paragraphs:
            if len(para) > 100:
                return para[:200].strip()
        return "No specific details found"

    def _extract_consequences(self, text: str) -> str:
        """Extract consequences of non-compliance"""
        consequence_patterns = [
            r'(?:failure|nonc|failure to comply|if you do not).*?(?:will result in|consequences|legal action|lawsuit)[:.\\s]*([^.!?]{30,200})',
            r'(?:consequences|penalties|damages)[:.\\s]*([^.!?]{30,200})',
            r'(?:we will|will be forced to|we intend to).*?(?:legal|court|lawsuit|damages)[:.\\s]*([^.!?]{40,200})',
        ]
        for pattern in consequence_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                return match.group(1).strip()
        return "Consequences not explicitly stated"

    def _extract_clauses(self, text: str) -> list:
        """Extract important clauses"""
        clauses = []
        clause_patterns = [
            r'(?:Section|Clause|Article)\s+(\d+)[:\s]+([^\n]{20,100})',
            r'^[-•*]\s+([^\n]{20,100})',
            r'(?:Clause|Section|Article)[:\s]+([^\n]{20,100})',
        ]
        for pattern in clause_patterns:
            matches = re.finditer(pattern, text, re.IGNORECASE | re.MULTILINE)
            for match in matches:
                clause_text = match.group(1) if match.lastindex and match.lastindex >= 1 else match.group(0)
                if clause_text and len(clause_text) > 10:
                    clauses.append(clause_text.strip()[:100])
        return clauses[:5]

    def _extract_signatures(self, text: str) -> list:
        """Extract signature locations"""
        signatures = []
        sig_patterns = [
            r'(?:Signed|Signature)[:\s]+(.+)',
            r'(?:By|Sign here)[:\s]+(.+)',
            r'_+\s+',
        ]
        for pattern in sig_patterns:
            matches = re.finditer(pattern, text, re.IGNORECASE)
            for match in matches:
                if match.lastindex:
                    signatures.append(f"Signature: {match.group(1).strip()[:50]}")
                else:
                    signatures.append("Signature block found")
        return signatures[:3]


# ========================================================================
# VALIDATION TOOL
# ========================================================================
class ValidationTool:
    """Tool for validating extracted document information"""

    def __init__(self):
        self.required_fields_by_type = {
            "Cease": ['sender', 'recipient', 'violation_type'],
            "Uncertain": ['sender'],
            "Generic": ['sender', 'recipient']
        }

    def validate_extraction(self, extracted_data: Dict[str, Any],
                           doc_classification: str = "Generic") -> Dict[str, Any]:
        """
        Validate extracted data quality

        Args:
            extracted_data: Dictionary of extracted fields
            doc_classification: Document classification type

        Returns:
            Dictionary with validation results
        """
        try:
            issues = []
            checks_passed = 0

            required_fields = self.required_fields_by_type.get(doc_classification, ['sender', 'recipient'])
            total_checks = len(required_fields)

            # Check required fields
            for field in required_fields:
                if extracted_data.get(field) and extracted_data.get(field) not in ["Unknown", ""]:
                    checks_passed += 1
                else:
                    issues.append(f"Missing or invalid field: {field}")

            # Additional quality checks
            for field, value in extracted_data.items():
                if isinstance(value, str) and len(value) > 5:
                    checks_passed += 0.2  # Bonus for non-empty string fields
                elif isinstance(value, (list, dict)) and len(value) > 0:
                    checks_passed += 0.2  # Bonus for non-empty complex fields

            confidence = min(100, (checks_passed / max(1, total_checks)) * 100)
            is_valid = checks_passed >= (total_checks * 0.6)  # 60% threshold

            recommendations = self._generate_recommendations(confidence, issues)

            return {
                "status": "SUCCESS",
                "is_valid": is_valid,
                "confidence_score": confidence,
                "checks_passed": checks_passed,
                "total_checks": total_checks,
                "issues_found": issues,
                "requires_review": confidence < 80,
                "recommendations": recommendations
            }

        except Exception as e:
            return {
                "status": "FAILED",
                "error": f"Validation error: {str(e)}",
                "is_valid": False,
                "confidence_score": 0.0,
                "issues_found": [str(e)],
                "requires_review": True
            }

    def _generate_recommendations(self, confidence: float, issues: list) -> list:
        """Generate recommendations based on validation"""
        recommendations = []

        if confidence < 60:
            recommendations.append("⚠ Recommend manual review due to low confidence score")

        if len(issues) > 0:
            recommendations.append(f"⚠ Address {len(issues)} validation issue(s)")

        if confidence >= 80:
            recommendations.append("✓ Data quality is good - proceed with processing")

        if "Missing field" in str(issues):
            recommendations.append("ℹ Request missing information")

        return recommendations


# ========================================================================
# INSTANTIATE TOOLS
# ========================================================================
pdf_loader_tool = PDFLoaderTool()
ocr_tool = OCRTool()
classification_tool = ClassificationTool()
extraction_tool = ExtractionTool()
validation_tool = ValidationTool()

print("✓ PDF Loader Tool initialized")
print("✓ OCR Tool initialized")
print("✓ Classification Tool initialized")
print("✓ Extraction Tool initialized")
print("✓ Validation Tool initialized")



✓ PDF Loader Tool initialized
✓ OCR Tool initialized
✓ Classification Tool initialized
✓ Extraction Tool initialized
✓ Validation Tool initialized


In [24]:
def create_document_processor():
    """Create the LangGraph workflow with classification routing and audit trail"""

    # Initialize Groq LLM
    groq_api_key = os.getenv("GROQ_API_KEY")
    if not groq_api_key:
        raise ValueError("GROQ_API_KEY environment variable not set")

    llm = ChatGroq(
        model="mixtral-8x7b-32768",
        temperature=0,
        groq_api_key=groq_api_key
    )

    # Create graph
    graph = StateGraph(ProcessingState)

    # ========================================================================
    # WORKFLOW STAGES WITH AUDIT LOGGING
    # ========================================================================

    def load_stage(state: ProcessingState) -> ProcessingState:
        """Stage 1: Load document using PDF Loader or OCR Tool with audit logging"""
        log_msg = f"[LOAD] Loading document: {state['file_path']}"
        state["processing_log"].append(log_msg)

        try:
            file_path = Path(state["file_path"])
            load_result = None
            tool_used = None

            # Use PDF Loader Tool for PDF files
            if file_path.suffix.lower() == '.pdf':
                load_result = pdf_loader_tool.load_pdf(state["file_path"])
                tool_used = "PDF Loader Tool"

            # Use OCR Tool for image files
            elif file_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff']:
                load_result = ocr_tool.extract_text_from_image(state["file_path"])
                tool_used = "OCR Tool"

            # Fallback for text files
            elif file_path.suffix.lower() in ['.txt', '.md']:
                with open(state["file_path"], 'r', encoding='utf-8') as f:
                    text = f.read()
                load_result = {
                    "status": "SUCCESS",
                    "text": text,
                    "page_count": 1,
                    "file_path": state["file_path"]
                }
                tool_used = "Text File Reader"

            else:
                raise ValueError(f"Unsupported file format: {file_path.suffix}")

            # Check if load was successful
            if load_result["status"] != "SUCCESS":
                raise ValueError(load_result.get("error", "Unknown load error"))

            text = load_result["text"]
            state["document_text"] = text

            # Audit: Document loaded successfully
            audit = audit_manager.create_audit_entry(
                doc_id=state["document_id"],
                operation_type="LOAD",
                status="SUCCESS",
                explanation=f"Document extracted using {tool_used}: {load_result.get('page_count', 'N/A')} pages",
                output_data={
                    "text_length": len(text),
                    "file_path": state["file_path"],
                    "tool_used": tool_used,
                    "page_count": load_result.get("page_count", 1)
                },
                confidence=load_result.get("confidence", 1.0),
                compliance_notes=f"Document successfully loaded using {tool_used}"
            )
            state["audit_trail"].append(audit)
            state["processing_log"].append(f"✓ Document loaded successfully using {tool_used}")

        except Exception as e:
            # Audit: Load failure
            audit = audit_manager.create_audit_entry(
                doc_id=state["document_id"],
                operation_type="LOAD",
                status="FAILURE",
                explanation=f"Document loading failed: {str(e)}",
                compliance_notes=f"Error: {str(e)}"
            )
            state["audit_trail"].append(audit)
            state["processing_log"].append(f"✗ Error loading document: {str(e)}")
            raise

        return state

    def classify_stage(state: ProcessingState) -> ProcessingState:
        """Stage 2: Classify document using Classification Tool with audit"""
        state["processing_log"].append("[CLASSIFY] Analyzing document classification using Classification Tool...")

        try:
            # Use Classification Tool
            result = classification_tool.classify(state["document_text"])

            if result["status"] != "SUCCESS":
                raise ValueError(result.get("error", "Classification tool error"))

            classification = result.get("classification", "Uncertain")
            confidence = result.get("confidence", 0)
            explanation = result.get("reasoning", "")

            state["document_classification"] = classification
            state["document_type"] = classification
            state["document_type_confidence"] = confidence

            # Audit: Classification decision
            audit = audit_manager.create_audit_entry(
                doc_id=state["document_id"],
                operation_type="CLASSIFY",
                status="SUCCESS",
                explanation=explanation,
                output_data={
                    "classification": classification,
                    "cease_score": result.get("cease_score", 0),
                    "uncertain_score": result.get("uncertain_score", 0),
                    "tool": "Classification Tool",
                    "reasoning": result.get("reasoning", "")
                },
                confidence=confidence,
                compliance_notes=f"Document classified as: {classification} with {confidence:.1f}% confidence using Classification Tool"
            )
            state["audit_trail"].append(audit)
            state["processing_log"].append(f"✓ Classification: {classification} ({confidence:.1f}%) [Classification Tool]")

        except Exception as e:
            # Audit: Classification failure
            audit = audit_manager.create_audit_entry(
                doc_id=state["document_id"],
                operation_type="CLASSIFY",
                status="FAILURE",
                explanation=f"Classification failed: {str(e)}",
                compliance_notes=f"Error: {str(e)}"
            )
            state["audit_trail"].append(audit)
            state["document_classification"] = "Uncertain"
            state["processing_log"].append(f"✗ Classification error: {str(e)}")

        return state

    def extract_stage(state: ProcessingState) -> ProcessingState:
        """Stage 3: Extract information using Extraction Tool based on classification"""
        state["processing_log"].append(f"[EXTRACT] Extracting data from {state['document_classification']} document using Extraction Tool...")

        try:
            # Use Extraction Tool
            extract_result = extraction_tool.extract_generic_document(
                state["document_text"],
                doc_classification=state["document_classification"]
            )

            if extract_result["status"] != "SUCCESS":
                raise ValueError(extract_result.get("error", "Extraction tool error"))

            data = extract_result.get("extracted_data", {})
            state["extracted_data"] = data
            field_count = extract_result.get("fields_extracted", 0)

            # Audit: Extraction success
            audit = audit_manager.create_audit_entry(
                doc_id=state["document_id"],
                operation_type="EXTRACT",
                status="SUCCESS",
                explanation=f"Extracted {field_count} data fields from {state['document_classification']} document using Extraction Tool",
                output_data={
                    "fields_extracted": field_count,
                    "extraction_type": state["document_classification"],
                    "extraction_method": extract_result.get("extraction_method", "regex"),
                    "tool": "Extraction Tool"
                },
                compliance_notes="Data extraction completed successfully using Extraction Tool"
            )
            state["audit_trail"].append(audit)
            state["processing_log"].append(f"✓ Extracted {field_count} fields using Extraction Tool")

        except Exception as e:
            # Audit: Extraction failure
            audit = audit_manager.create_audit_entry(
                doc_id=state["document_id"],
                operation_type="EXTRACT",
                status="FAILURE",
                explanation=f"Extraction failed: {str(e)}",
                compliance_notes=f"Error: {str(e)}"
            )
            state["audit_trail"].append(audit)
            state["processing_log"].append(f"✗ Extraction error: {str(e)}")
            state["extracted_data"] = {}

        return state

    def validate_stage(state: ProcessingState) -> ProcessingState:
        """Stage 4: Validate extracted information using Validation Tool"""
        state["processing_log"].append("[VALIDATE] Validating extracted data using Validation Tool...")

        try:
            # Use Validation Tool
            validation_result = validation_tool.validate_extraction(
                state["extracted_data"],
                doc_classification=state["document_classification"]
            )

            if validation_result["status"] != "SUCCESS":
                raise ValueError(validation_result.get("error", "Validation tool error"))

            validation = ValidationResult(
                is_valid=validation_result.get("is_valid", False),
                confidence_score=validation_result.get("confidence_score", 0.0),
                issues_found=validation_result.get("issues_found", []),
                requires_review=validation_result.get("requires_review", True),
                recommendations=validation_result.get("recommendations", [])
            )

            state["validation_result"] = validation
            state["confidence_score"] = validation.confidence_score
            state["requires_human_review"] = validation.requires_review

            # Audit: Validation result
            audit = audit_manager.create_audit_entry(
                doc_id=state["document_id"],
                operation_type="VALIDATE",
                status="SUCCESS" if validation.is_valid else "WARNING",
                explanation=f"Validation for {state['document_classification']} document: {len(validation.issues_found)} issues found using Validation Tool",
                output_data={
                    "is_valid": validation.is_valid,
                    "issues": validation.issues_found,
                    "tool": "Validation Tool",
                    "checks_passed": validation_result.get("checks_passed", 0),
                    "total_checks": validation_result.get("total_checks", 0)
                },
                confidence=validation.confidence_score,
                compliance_notes=f"Document validation complete using Validation Tool. Requires review: {validation.requires_review}"
            )
            state["audit_trail"].append(audit)
            state["processing_log"].append(f"✓ Validation: confidence {validation.confidence_score:.1f}% [Validation Tool]")

        except Exception as e:
            audit = audit_manager.create_audit_entry(
                doc_id=state["document_id"],
                operation_type="VALIDATE",
                status="FAILURE",
                explanation=f"Validation failed: {str(e)}",
                compliance_notes=f"Error: {str(e)}"
            )
            state["audit_trail"].append(audit)
            state["processing_log"].append(f"✗ Validation error: {str(e)}")

        return state

    def route_stage(state: ProcessingState) -> ProcessingState:
        """Stage 5: Route to appropriate agent based on classification"""
        classification = state["document_classification"]
        state["processing_log"].append(f"[ROUTE] Routing {classification} document to appropriate handler...")

        received_date = state.get("received_date", datetime.now().isoformat())
        doc_name = Path(state["file_path"]).name

        try:
            if classification == "Cease":
                # Route to Database Agent
                state["processing_log"].append("→ Routing to DATABASE AGENT for storage")
                result = database_agent.store_cease_document(
                    doc_id=state["document_id"],
                    received_date=received_date,
                    doc_name=doc_name,
                    extracted=state["extracted_data"],
                    confidence=state["confidence_score"],
                    audit_id=state["audit_trail"][-1].audit_id if state["audit_trail"] else "unknown"
                )

                audit = audit_manager.create_audit_entry(
                    doc_id=state["document_id"],
                    operation_type="ROUTE",
                    status="SUCCESS",
                    explanation=f"Cease document routed to database storage. Status: {result['status']}",
                    output_data=result,
                    compliance_notes="Cease & Desist document stored in database with full audit trail"
                )
                state["routing_decision"] = "DATABASE_STORED"
                state["processing_log"].append(f"✓ {result['message']}")

            elif classification == "Irrelevant":
                # Route to Archiving Agent
                state["processing_log"].append("→ Routing to ARCHIVING AGENT for flat-file storage")
                result = archiving_agent.archive_document(
                    doc_id=state["document_id"],
                    received_date=received_date,
                    doc_name=doc_name,
                    doc_text=state["document_text"],
                    audit_id=state["audit_trail"][-1].audit_id if state["audit_trail"] else "unknown"
                )

                audit = audit_manager.create_audit_entry(
                    doc_id=state["document_id"],
                    operation_type="ROUTE",
                    status="SUCCESS",
                    explanation=f"Irrelevant document archived. Status: {result['status']}",
                    output_data=result,
                    compliance_notes="Non-cease document archived to flat file"
                )
                state["routing_decision"] = "ARCHIVED"
                state["processing_log"].append(f"✓ {result['message']}")

            elif classification == "Uncertain":
                # Route to HITL Review Agent
                state["processing_log"].append("→ Routing to HUMAN-IN-THE-LOOP REVIEW")
                result = hitl_agent.queue_for_review(
                    doc_id=state["document_id"],
                    received_date=received_date,
                    doc_name=doc_name,
                    extracted=state["extracted_data"],
                    confidence=state["confidence_score"],
                    audit_id=state["audit_trail"][-1].audit_id if state["audit_trail"] else "unknown"
                )

                audit = audit_manager.create_audit_entry(
                    doc_id=state["document_id"],
                    operation_type="ROUTE",
                    status=result['status'],
                    explanation=f"Uncertain document queued for manual review. Location: {result.get('review_location', 'unknown')}",
                    output_data=result,
                    compliance_notes="Document requires human review before final classification"
                )
                state["routing_decision"] = "PENDING_HUMAN_REVIEW"
                state["processing_log"].append(f"✓ {result['message']}")

            state["audit_trail"].append(audit)

        except Exception as e:
            audit = audit_manager.create_audit_entry(
                doc_id=state["document_id"],
                operation_type="ROUTE",
                status="FAILURE",
                explanation=f"Routing failed: {str(e)}",
                compliance_notes=f"Error during document routing: {str(e)}"
            )
            state["audit_trail"].append(audit)
            state["processing_log"].append(f"✗ Routing error: {str(e)}")
            state["routing_decision"] = "ERROR"

        return state

    def finalize_stage(state: ProcessingState) -> ProcessingState:
        """Stage 6: Create final result with complete audit trail"""
        state["processing_log"].append("[FINALIZE] Creating final result with audit trail...")

        try:
            metadata = DocumentMetadata(
                document_id=state["document_id"],
                document_type=state["document_classification"],
                classification=state["document_classification"],
                processing_date=datetime.now().isoformat(),
                confidence_score=state["confidence_score"],
                file_path=state["file_path"],
                received_date=state.get("received_date", datetime.now().isoformat())
            )

            result = DocumentProcessingResult(
                metadata=metadata,
                document_type=state["document_classification"],
                classification=state["document_classification"],
                extracted_data=state["extracted_data"],
                validation=state["validation_result"],
                raw_text=state["document_text"][:1000],
                processing_log=state["processing_log"],
                audit_trail=state["audit_trail"],
                routing_decision=state.get("routing_decision", "PENDING")
            )

            state["final_result"] = result

            # Final audit: Processing complete
            audit = audit_manager.create_audit_entry(
                doc_id=state["document_id"],
                operation_type="PROCESSING_COMPLETE",
                status="SUCCESS",
                explanation=f"Document processing complete. Classification: {state['document_classification']}, Routing: {state.get('routing_decision', 'pending')}",
                output_data={"routing_decision": state.get("routing_decision")},
                compliance_notes="Full processing pipeline completed with comprehensive audit trail"
            )
            state["audit_trail"].append(audit)
            state["processing_log"].append("✓ Processing complete with full audit trail")

        except Exception as e:
            state["processing_log"].append(f"✗ Finalization error: {str(e)}")

        return state

    # Add nodes to graph
    graph.add_node("load", load_stage)
    graph.add_node("classify", classify_stage)
    graph.add_node("extract", extract_stage)
    graph.add_node("validate", validate_stage)
    graph.add_node("route", route_stage)
    graph.add_node("finalize", finalize_stage)

    # Add edges - linear workflow
    graph.add_edge(START, "load")
    graph.add_edge("load", "classify")
    graph.add_edge("classify", "extract")
    graph.add_edge("extract", "validate")
    graph.add_edge("validate", "route")
    graph.add_edge("route", "finalize")
    graph.add_edge("finalize", END)

    # Compile with in-memory persistence
    from langgraph.checkpoint.memory import MemorySaver

    checkpointer = MemorySaver()
    workflow = graph.compile(checkpointer=checkpointer)

    return workflow

print("✓ LangGraph workflow created successfully!")
print("✓ Tools integrated into workflow:")
print("  • PDF Loader Tool → Load Stage")
print("  • OCR Tool → Load Stage (for images)")
print("  • Classification Tool → Classify Stage")
print("  • Extraction Tool → Extract Stage")
print("  • Validation Tool → Validate Stage")
print("✓ Classification routing: Cease → Database")
print("✓                        Irrelevant → Archive")
print("✓                        Uncertain → Human Review")
print("✓ Full audit trail enabled with tool tracking")

✓ LangGraph workflow created successfully!
✓ Tools integrated into workflow:
  • PDF Loader Tool → Load Stage
  • OCR Tool → Load Stage (for images)
  • Classification Tool → Classify Stage
  • Extraction Tool → Extract Stage
  • Validation Tool → Validate Stage
✓ Classification routing: Cease → Database
✓                        Irrelevant → Archive
✓                        Uncertain → Human Review
✓ Full audit trail enabled with tool tracking


In [25]:
# Discover PDF documents from the guides folder

from pathlib import Path
import glob

# Path to the PDFs folder
GUIDES_PDF_PATH = r"/content/sample_data/Data"

def discover_pdf_documents(pdf_folder):
    """
    Discover all PDF files in the specified folder
    """
    pdf_path = Path(pdf_folder)

    if not pdf_path.exists():
        print(f"❌ Error: PDF folder not found at {pdf_folder}")
        return []

    # Find all PDF files
    pdf_files = sorted(pdf_path.glob("*.pdf"))

    if not pdf_files:
        print(f"❌ Error: No PDF files found in {pdf_folder}")
        return []

    print(f"✓ Found {len(pdf_files)} PDF documents:")
    print("-" * 60)

    documents = []
    for i, pdf_file in enumerate(pdf_files, 1):
        doc_id = f"PDF_{i:03d}_{pdf_file.stem}"
        documents.append({
            "file_path": str(pdf_file),
            "filename": pdf_file.name,
            "doc_id": doc_id,
            "index": i
        })
        print(f"{i:2d}. {pdf_file.name} → {doc_id}")

    print("-" * 60)
    print(f"✓ Total documents to process: {len(documents)}\n")

    return documents

# Discover the PDFs
discovered_documents = discover_pdf_documents(GUIDES_PDF_PATH)

if discovered_documents:
    print(f"✓ Successfully discovered {len(discovered_documents)} PDF documents")
    print("Ready to process documents in the next step!")
else:
    print("⚠ No documents found. Please check the PDF folder path.")

✓ Found 29 PDF documents:
------------------------------------------------------------
 1. 01_copyright_infringement_photography.pdf → PDF_001_01_copyright_infringement_photography
 2. 02_trademark_infringement_tech.pdf → PDF_002_02_trademark_infringement_tech
 3. 03_trade_secret_misappropriation.pdf → PDF_003_03_trade_secret_misappropriation
 4. 04_defamation_online_review.pdf → PDF_004_04_defamation_online_review
 5. 05_patent_infringement_medical_device.pdf → PDF_005_05_patent_infringement_medical_device
 6. 06_harassment_workplace.pdf → PDF_006_06_harassment_workplace
 7. 07_software_license_violation.pdf → PDF_007_07_software_license_violation
 8. 08_non_compete_violation.pdf → PDF_008_08_non_compete_violation
 9. 09_copyright_infringement_music.pdf → PDF_009_09_copyright_infringement_music
10. 10_breach_of_contract_nda.pdf → PDF_010_10_breach_of_contract_nda
11. LOA2.pdf → PDF_011_LOA2
12. LOA3.pdf → PDF_012_LOA3
13. LOA4.pdf → PDF_013_LOA4
14. LOA5.pdf → PDF_014_LOA5
15. LOA6.pd

In [26]:
def process_document(file_path: str, document_id: str = None, received_date: str = None) -> DocumentProcessingResult:
    """
    Process a document using the LangGraph workflow with classification and audit trail.

    Args:
        file_path: Path to document file
        document_id: Optional unique ID for document
        received_date: Optional date document was received

    Returns:
        DocumentProcessingResult with extraction, validation, and audit trail
    """
    if document_id is None:
        document_id = f"doc_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    if received_date is None:
        received_date = datetime.now().isoformat()

    print(f"\n{'='*70}")
    print(f"Processing Document: {document_id}")
    print(f"File: {file_path}")
    print(f"Received: {received_date}")
    print(f"{'='*70}\n")

    # Create workflow
    workflow = create_document_processor()

    # Initialize state with audit trail
    initial_state = {
        "document_id": document_id,
        "file_path": file_path,
        "received_date": received_date,
        "document_text": "",
        "document_type": "",
        "document_classification": "Uncertain",
        "document_type_confidence": 0.0,
        "extracted_data": {},
        "validation_result": None,
        "confidence_score": 0.0,
        "processing_log": [f"[INIT] Processing started for {document_id}"],
        "audit_trail": [],
        "requires_human_review": False,
        "human_review_result": None,
        "final_result": None,
        "messages": [],
        "is_approved": False
    }

    # Run workflow
    config = {
        "configurable": {
            "thread_id": document_id
        }
    }

    result_state = workflow.invoke(initial_state, config)

    # Print results
    print("\n" + "="*70)
    print("PROCESSING LOG")
    print("="*70)
    for log_entry in result_state["processing_log"]:
        print(log_entry)

    print("\n" + "="*70)
    print("CLASSIFICATION RESULT")
    print("="*70)
    print(f"Classification: {result_state['document_classification']}")
    print(f"Confidence: {result_state['confidence_score']:.1f}%")
    print(f"Routing Decision: {result_state.get('routing_decision', 'PENDING')}")

    print("\nExtracted Data:")
    print("-" * 70)
    print(json.dumps(result_state["extracted_data"], indent=2, default=str))

    if result_state["validation_result"]:
        print("\nValidation Details:")
        print("-" * 70)
        print(f"Valid: {result_state['validation_result'].is_valid}")
        print(f"Confidence: {result_state['validation_result'].confidence_score:.1f}%")
        if result_state["validation_result"].issues_found:
            print("Issues Found:")
            for issue in result_state["validation_result"].issues_found:
                print(f"  - {issue}")
        if result_state["validation_result"].recommendations:
            print("Recommendations:")
            for rec in result_state["validation_result"].recommendations:
                print(f"  - {rec}")

    # Print audit trail
    print("\n" + "="*70)
    print("AUDIT TRAIL")
    print("="*70)
    audit_manager.print_audit_trail(document_id)

    print("\n" + "="*70)

    return result_state["final_result"]

print("✓ Enhanced process_document function ready!")

✓ Enhanced process_document function ready!


In [27]:
# Process all discovered PDF documents

print("\n🔄 PROCESSING PDF DOCUMENTS FROM GUIDES FOLDER\n")

# Store results for summary
processing_results = []
failed_documents = []

if discovered_documents:
    total_docs = len(discovered_documents)

    for i, doc_info in enumerate(discovered_documents, 1):
        try:
            print(f"\n{'='*70}")
            print(f"Processing {i}/{total_docs}: {doc_info['filename']}")
            print(f"{'='*70}")

            result = process_document(doc_info["file_path"], doc_info["doc_id"])

            if result:
                processing_results.append({
                    "filename": doc_info["filename"],
                    "doc_id": doc_info["doc_id"],
                    "result": result,
                    "status": "SUCCESS"
                })
                print(f"✓ Successfully processed: {doc_info['filename']}")
            else:
                failed_documents.append(doc_info["filename"])
                print(f"⚠ Failed to process: {doc_info['filename']}")

        except Exception as e:
            failed_documents.append(doc_info["filename"])
            print(f"❌ Error processing {doc_info['filename']}: {str(e)}")

    print(f"\n✓ Batch processing complete!")
    print(f"   Successfully processed: {len(processing_results)}/{total_docs}")
    if failed_documents:
        print(f"   Failed: {len(failed_documents)}")
else:
    print("❌ No documents to process. Please run Step 8 first.")


🔄 PROCESSING PDF DOCUMENTS FROM GUIDES FOLDER


Processing 1/29: 01_copyright_infringement_photography.pdf

Processing Document: PDF_001_01_copyright_infringement_photography
File: /content/sample_data/Data/01_copyright_infringement_photography.pdf
Received: 2026-03-26T11:19:18.316942


PROCESSING LOG
[INIT] Processing started for PDF_001_01_copyright_infringement_photography
[LOAD] Loading document: /content/sample_data/Data/01_copyright_infringement_photography.pdf
✓ Document loaded successfully using PDF Loader Tool
[CLASSIFY] Analyzing document classification using Classification Tool...
✓ Classification: Cease (100.0%) [Classification Tool]
[EXTRACT] Extracting data from Cease document using Extraction Tool...
✓ Extracted 7 fields using Extraction Tool
[VALIDATE] Validating extracted data using Validation Tool...
✓ Validation: confidence 100.0% [Validation Tool]
[ROUTE] Routing Cease document to appropriate handler...
→ Routing to DATABASE AGENT for storage
✓ ✓ Cease document sto

In [28]:
# Summary of all processed documents

print("\n" + "="*70)
print("BATCH PROCESSING SUMMARY")
print("="*70)

if processing_results:
    summary_data = []

    for item in processing_results:
        result = item["result"]
        if result:
            summary_data.append({
                "Filename": item["filename"],
                "Document ID": item["doc_id"],
                "Type": result.document_type,
                "Confidence": f"{result.validation.confidence_score:.1f}%",
                "Valid": result.validation.is_valid,
                "Requires Review": result.validation.requires_review
            })

            print(f"\n📄 {item['filename']}")
            print(f"   ID: {item['doc_id']}")
            print(f"   Type: {result.document_type}")
            print(f"   Confidence: {result.validation.confidence_score:.1f}%")
            print(f"   Valid: {result.validation.is_valid}")
            print(f"   Requires Review: {result.validation.requires_review}")

    print("\n" + "="*70)
    print(f"✓ Successfully processed {len(processing_results)} documents!")
    print("="*70)
else:
    print("\n❌ No documents were processed. Please check for errors above.")
    print("="*70)


BATCH PROCESSING SUMMARY

📄 01_copyright_infringement_photography.pdf
   ID: PDF_001_01_copyright_infringement_photography
   Type: Cease
   Confidence: 100.0%
   Valid: True
   Requires Review: False

📄 02_trademark_infringement_tech.pdf
   ID: PDF_002_02_trademark_infringement_tech
   Type: Cease
   Confidence: 100.0%
   Valid: True
   Requires Review: False

📄 03_trade_secret_misappropriation.pdf
   ID: PDF_003_03_trade_secret_misappropriation
   Type: Cease
   Confidence: 100.0%
   Valid: True
   Requires Review: False

📄 04_defamation_online_review.pdf
   ID: PDF_004_04_defamation_online_review
   Type: Cease
   Confidence: 100.0%
   Valid: True
   Requires Review: False

📄 05_patent_infringement_medical_device.pdf
   ID: PDF_005_05_patent_infringement_medical_device
   Type: Cease
   Confidence: 100.0%
   Valid: True
   Requires Review: False

📄 06_harassment_workplace.pdf
   ID: PDF_006_06_harassment_workplace
   Type: Cease
   Confidence: 100.0%
   Valid: True
   Requires Revi

In [29]:
# Export results to JSON

export_data = {
    "processing_date": datetime.now().isoformat(),
    "pdf_folder": GUIDES_PDF_PATH,
    "total_documents_found": len(discovered_documents),
    "documents_processed": len(processing_results),
    "documents": []
}

for item in processing_results:
    result = item["result"]
    if result:
        export_data["documents"].append({
            "filename": item["filename"],
            "doc_id": item["doc_id"],
            "document_type": result.document_type,
            "confidence": result.validation.confidence_score,
            "is_valid": result.validation.is_valid,
            "requires_review": result.validation.requires_review,
            "extracted_data": result.extracted_data,
            "validation_issues": result.validation.issues_found,
            "recommendations": result.validation.recommendations,
            "processing_log": result.processing_log
        })

# Save to JSON file
output_file = "document_processing_results.json"
with open(output_file, 'w') as f:
    json.dump(export_data, f, indent=2, default=str)

print(f"✓ Results exported to {output_file}")
print(f"Total documents processed: {export_data['documents_processed']}/{export_data['total_documents_found']}")

✓ Results exported to document_processing_results.json
Total documents processed: 29/29
